# Complaints agent — escalation and severity models (real complaints)

**What it learns, from 100 000+ real consumer complaints (US CFPB database):**
1. **Escalation** — will the customer *dispute* the company's answer? (`Consumer disputed?` = Yes) → the agent's *escalation risk*.
2. **Severity** — will the case end with *money paid back* to the customer? (`Company Response = Closed with monetary relief`) → the agent's *severity*.

**Model:** TF-IDF + logistic regression. Chosen on purpose: every prediction can be explained word by word (our explainable-AI promise), it runs anywhere on CPU, and it is exported as plain JSON (no library-version problems).
**Data:** [milesbutler/consumer_complaints](https://huggingface.co/datasets/milesbutler/consumer_complaints) — the public CFPB Consumer Complaint Database (US public data). Domain = financial services, not telecom: we use it for *how* customers write when a complaint will escalate, not for telecom facts.

**Run:** CPU is enough (GPU useless here). ~10–20 min. Download **`complaints_models.zip`** at the end.

In [ ]:
!pip install -q -U datasets scikit-learn pyarrow

In [ ]:
import os, json, re, random, shutil, time
from pathlib import Path
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED); np.random.seed(SEED)
# Kaggle writes to /kaggle/working, Lightning AI to the current folder
OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
print("Outputs go to", OUT)

In [ ]:
from datasets import load_dataset
ds = load_dataset("milesbutler/consumer_complaints")
df = pd.concat([ds[s].to_pandas() for s in ds.keys()], ignore_index=True)
print(df.shape); print(list(df.columns))

def col(*words):
    """Find a column whatever its exact spelling ('Consumer disputed?' / 'consumer_disputed')."""
    norm = {c: re.sub(r"[^a-z]", "", c.lower()) for c in df.columns}
    for c, n in norm.items():
        if all(w in n for w in words): return c
    raise KeyError(words)

TEXT, DISPUTED, RESPONSE = col("complaint", "consumer") if "Consumer Complaint" not in df.columns else "Consumer Complaint", col("disputed"), col("response", "consumer")
print("text:", TEXT, "| disputed:", DISPUTED, "| response:", RESPONSE)
df = df[df[TEXT].notna()].copy()
df["text"] = (df[TEXT].astype(str)
              .str.replace(r"X{2,}", " ", regex=True)            # CFPB redactions: XXXX
              .str.replace(r"\{\$[\d.,]+\}", " AMOUNT ", regex=True)
              .str.replace(r"\s+", " ", regex=True).str.strip())
df = df[df.text.str.len() >= 30]
print(len(df), "complaints with a narrative")
print(df[DISPUTED].value_counts(dropna=False).head()); print(df[RESPONSE].value_counts(dropna=False).head(10))

## The two targets

In [ ]:
esc = df[df[DISPUTED].isin(["Yes", "No"])].copy()
esc["y"] = (esc[DISPUTED] == "Yes").astype(int)

resp = df[RESPONSE].fillna("")
sev = df[resp.str.startswith("Closed")].copy()
r = sev[RESPONSE].str.lower()
sev["y"] = (r.str.contains("monetary relief") & ~r.str.contains("non-monetary")).astype(int)

for name, d in [("escalation (disputed)", esc), ("severity (monetary relief)", sev)]:
    print(f"{name}: {len(d)} rows, positives {d.y.mean():.1%}")

## Training: TF-IDF + logistic regression, class-balanced, 80/20 stratified split

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_recall_curve

MAX_ROWS = 200_000
VEC = dict(ngram_range=(1, 2), min_df=5, max_df=0.5, max_features=20_000, sublinear_tf=True, stop_words="english", lowercase=True)

def train(d, name):
    d = d.sample(min(len(d), MAX_ROWS), random_state=SEED)
    tr, te = train_test_split(d, test_size=0.2, stratify=d.y, random_state=SEED)
    vec = TfidfVectorizer(**VEC)
    Xtr, Xte = vec.fit_transform(tr.text), vec.transform(te.text)
    clf = LogisticRegression(max_iter=3000, class_weight="balanced", C=1.0, solver="liblinear").fit(Xtr, tr.y)
    p = clf.predict_proba(Xte)[:, 1]
    prec, rec, thr = precision_recall_curve(te.y, p)
    f1s = 2 * prec * rec / np.clip(prec + rec, 1e-9, None)
    best = int(np.nanargmax(f1s[:-1]))
    m = dict(rows_train=len(tr), rows_test=len(te), positive_rate=float(te.y.mean()),
             roc_auc=float(roc_auc_score(te.y, p)), pr_auc=float(average_precision_score(te.y, p)),
             best_threshold=float(thr[best]), f1_at_best=float(f1s[best]), f1_at_0_5=float(f1_score(te.y, p >= 0.5)))
    print(f"\n== {name}"); print(json.dumps(m, indent=2))
    terms = np.array(vec.get_feature_names_out()); w = clf.coef_[0]
    print("words that RAISE it :", ", ".join(terms[np.argsort(w)[-25:][::-1]]))
    print("words that LOWER it :", ", ".join(terms[np.argsort(w)[:25]]))
    return vec, clf, m

esc_vec, esc_clf, esc_m = train(esc, "ESCALATION (will the customer dispute?)")
sev_vec, sev_clf, sev_m = train(sev, "SEVERITY (will money be paid back?)")

## Sanity check on telecom-style complaints (like the ones our agent receives)

In [ ]:
TELECOM = [
 "I was charged twice for my mobile plan and nobody answers, I will go to the regulator and post this everywhere",
 "My internet has been down for a week, I called five times, I demand a refund for every day without service",
 "Small question about my invoice, the amount seems a bit higher than usual",
 "You promised a credit last month and it never came, this is fraud, my lawyer will contact you",
 "Thank you for fixing the roaming issue, I just want a copy of the invoice",
]
for t in TELECOM:
    print(f"escalation {esc_clf.predict_proba(esc_vec.transform([t]))[0,1]:.2f}  severity {sev_clf.predict_proba(sev_vec.transform([t]))[0,1]:.2f}  | {t[:80]}")

## Export as plain JSON (vocabulary, idf, weights) — rebuilt in the project with any scikit-learn version

In [ ]:
EXPORT = OUT / "complaints_models"; shutil.rmtree(EXPORT, ignore_errors=True); EXPORT.mkdir()
def dump(vec, clf, m, name, target):
    json.dump(dict(target=target, vectorizer={k: (list(v) if isinstance(v, tuple) else v) for k, v in VEC.items()},
                   vocabulary={t: int(i) for t, i in vec.vocabulary_.items()}, idf=[float(x) for x in vec.idf_],
                   coef=[float(x) for x in clf.coef_[0]], intercept=float(clf.intercept_[0]), metrics=m,
                   data="milesbutler/consumer_complaints (US CFPB Consumer Complaint Database, public)"),
              open(EXPORT / f"{name}.json", "w"))
dump(esc_vec, esc_clf, esc_m, "escalation", "Consumer disputed? = Yes")
dump(sev_vec, sev_clf, sev_m, "severity", "Company response = Closed with monetary relief")
(EXPORT / "MODEL_CARD.md").write_text(f'''# Complaints models (CFPB)
Escalation: ROC-AUC {esc_m["roc_auc"]:.3f}, PR-AUC {esc_m["pr_auc"]:.3f} (positives {esc_m["positive_rate"]:.1%}).
Severity: ROC-AUC {sev_m["roc_auc"]:.3f}, PR-AUC {sev_m["pr_auc"]:.3f} (positives {sev_m["positive_rate"]:.1%}).
TF-IDF (1-2 grams, 20k terms) + class-balanced logistic regression, 80/20 stratified split.
Limits: US financial complaints, English; proxies (dispute, monetary relief) for escalation and severity; not telecom-specific.
''')
shutil.make_archive(str(OUT / "complaints_models"), "zip", EXPORT)
print("Download:", OUT / "complaints_models.zip", f"({(OUT / 'complaints_models.zip').stat().st_size/1e6:.1f} MB)")